## Q1. Build Your Personalized Knowledge Base

In [1]:
import pandas as pd


roll_number = input("Enter your college roll number: ").strip()


last_two = roll_number[-2:]


categories = ["billing", "account", "general"]

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

personalized_entries = []

for d in last_two:
    digit = int(d)
    category = categories[digit % 3]

    if category == "billing":
        question = "how can i check my fee payment status"
        answer = "You can check your fee payment status in the billing section."
        keywords = "fee payment status billing"
    elif category == "account":
        question = "how do i update my registered mobile number"
        answer = "Go to Account Settings and update your registered mobile number."
        keywords = "mobile number update account"
    else:  # general
        question = "where can i find general college information"
        answer = "You can find general college information on the college portal."
        keywords = "college information general portal"

    personalized_entries.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })

faq_entries = fixed_entries + personalized_entries

df = pd.DataFrame(faq_entries)

print("Last two digits:", last_two)
print("\nFinal 6-row DataFrame:")
display(df)

Last two digits: 37

Final 6-row DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my fee payment status,You can check your fee payment status in the b...,fee payment status billing,billing
5,how do i update my registered mobile number,Go to Account Settings and update your registe...,mobile number update account,account


## Q2. Generate and Score a Hypothesis

In [2]:
def score_query(query, df):
    """
    Return all matching FAQ entries ranked by confidence.

    Score = number of query words that occur in the entry's
    question or keywords.
    """
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        text_words = set(text.split())

        score = len(query_words.intersection(text_words))

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "keywords": row["keywords"],
                "category": row["category"],
                "confidence": score
            })

    result_df = pd.DataFrame(results)

    if not result_df.empty:
        result_df = result_df.sort_values(
            by="confidence", ascending=False
        ).reset_index(drop=True)

    return result_df



query = input("Enter a query: ")
print("\nMatching entries ranked by confidence:")
display(score_query(query, df))


Matching entries ranked by confidence:


,question,answer,keywords,category,confidence
0,how do i update my registered mobile number,Go to Account Settings and update your registe...,mobile number update account,account,1


## Q3. Find All Questions in the Same Category

In [3]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()][
        ["question", "answer", "keywords", "category"]
    ]



personalized_category = personalized_entries[0]["category"]

print("Category selected:", personalized_category)
print("\nQuestions belonging to this category:")
display(same_category(personalized_category, df))

Category selected: billing

Questions belonging to this category:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my fee payment status,You can check your fee payment status in the b...,fee payment status billing,billing


## Q4. Add a New Keyword and Save the DataFrame

In [4]:

entry_index = 0

print("Selected FAQ:")
print(df.loc[entry_index, "question"])

new_keyword = input("Enter a new keyword to add: ").strip().lower()

if new_keyword:
    existing_keywords = df.loc[entry_index, "keywords"].split()

    if new_keyword not in existing_keywords:
        existing_keywords.append(new_keyword)

    df.loc[entry_index, "keywords"] = " ".join(existing_keywords)



csv_filename = f"{roll_number}_faq_data.csv"
df.to_csv(csv_filename, index=False)

print("\nUpdated DataFrame:")
display(df)

print(f"Saved successfully as: {csv_filename}")

Selected FAQ:
what is the annual fee

Updated DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge helloo,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how can i check my fee payment status,You can check your fee payment status in the b...,fee payment status billing,billing
5,how do i update my registered mobile number,Go to Account Settings and update your registe...,mobile number update account,account


Saved successfully as: 1024170437_faq_data.csv


## Q5. Count FAQ Entries Per Category

In [5]:
category_counts = df.groupby("category").size()

print("Number of FAQ entries per category:")
print(category_counts)

Number of FAQ entries per category:
category
account    2
billing    3
general    1
dtype: int64


## Q6. Modified Scoring Function — Handle Ties

In [6]:
def score_query_with_ties(query, df):
    """
    Score all matching entries and explicitly show all entries
    having the highest score.
    """
    result_df = score_query(query, df)

    if result_df.empty:
        print("No matching entries found.")
        return result_df

    highest_score = result_df["confidence"].max()
    top_matches = result_df[result_df["confidence"] == highest_score]

    print(f"Highest confidence score: {highest_score}")

    if len(top_matches) > 1:
        print("\nTIE: Multiple entries have the highest score.")
        print("All equally good matches:")
    else:
        print("\nNo tie. Best match:")

    display(top_matches)

    return result_df



print("TIE DEMONSTRATION")
tie_query = "fee"
tie_results = score_query_with_ties(tie_query, df)

print("\nNON-TIE DEMONSTRATION")
non_tie_query = "password reset"
non_tie_results = score_query_with_ties(non_tie_query, df)

TIE DEMONSTRATION
Highest confidence score: 1

TIE: Multiple entries have the highest score.
All equally good matches:


,question,answer,keywords,category,confidence
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge helloo,billing,1
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,1
2,how can i check my fee payment status,You can check your fee payment status in the b...,fee payment status billing,billing,1



NON-TIE DEMONSTRATION
Highest confidence score: 2

No tie. Best match:


,question,answer,keywords,category,confidence
0,how to reset password,Go to Settings > Reset Password.,password reset login,account,2
